# SenpaiQC: train the YOLO fast path (Colab)

Trains the `yolo11n-cls` classifier that answers the easy parts on Cloud Run CPU, so only
uncertain or new parts escalate to Gemini Robotics ER 2.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. On your machine: `python data/prepare_casting.py` (done once), which writes
   `data/colab/senpaiqc_data.zip` (train/val split + the held-out golden set).
3. Either upload that zip when cell 3 asks, or put it in Google Drive as
   `MyDrive/senpaiqc_data.zip` (faster on a slow connection).

**Out:** `senpaiqc_yolo.zip` with `best.pt`, `yolo_label_counts.json`, `yolo_baseline.json`
and `golden_yolo.json`. Unzip it in the repo root (see the last cell).

The golden set is **never** used for training or for choosing epochs; it is only scored at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"
%pip install -q ultralytics

## Settings

In [ ]:
EPOCHS = 15          # same default as eval/train_yolo.py
IMGSZ = 224
PART_NUMBER = "IMP-200"   # the part the seed cards and the app use
# Must match server/api/domain/policy.py EscalationPolicy
KEEP_THRESHOLD, ESCALATE_FLOOR = 0.85, 0.40

## Get the data

In [ ]:
import os, zipfile, pathlib
ZIP = "/content/senpaiqc_data.zip"
drive_zip = "/content/drive/MyDrive/senpaiqc_data.zip"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)
if os.path.exists(drive_zip):
    ZIP = drive_zip
    print("Using", ZIP)
elif not os.path.exists(ZIP):
    from google.colab import files
    print("Upload data/colab/senpaiqc_data.zip")
    up = files.upload()
    ZIP = "/content/" + next(iter(up))
DATA = pathlib.Path("/content/data")
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DATA)
for d in sorted((DATA / "splits/classify").glob("*/*")):
    print(d.relative_to(DATA), len(list(d.iterdir())))
print("golden", len(list((DATA / "golden").rglob("*.jp*g"))))

## Train

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n-cls.pt")
model.train(data=str(DATA / "splits/classify"), epochs=EPOCHS, imgsz=IMGSZ, batch=64, seed=0,
            project="/content/runs", name="baseline", exist_ok=True)
BEST = "/content/runs/baseline/weights/best.pt"
val = YOLO(BEST).val(data=str(DATA / "splits/classify"), split="val", imgsz=IMGSZ)
print("val top1:", round(float(val.top1), 4))

## Score the held-out golden set, with the app's escalation policy

Latency is measured on **CPU** (as Cloud Run serves it), not on the Colab GPU.

In [ ]:
import csv, json, time
best = YOLO(BEST)
rows = list(csv.DictReader(open(DATA / "golden/manifest.csv")))
res = []
for r in rows:
    t = time.perf_counter()
    p = best.predict(str(DATA / "golden" / r["path"]), device="cpu", imgsz=IMGSZ, verbose=False)[0]
    ms = (time.perf_counter() - t) * 1000
    cls, conf = p.names[int(p.probs.top1)], float(p.probs.top1conf)
    res.append({"path": r["path"], "label": r["label"], "pred": cls, "conf": round(conf, 4), "ms": round(ms, 1)})

def rate(xs, f):
    return round(sum(1 for x in xs if f(x)) / len(xs), 4) if xs else None

defective = [x for x in res if x["label"] == "defective"]
ok = [x for x in res if x["label"] == "ok"]
kept = [x for x in res if x["conf"] >= KEEP_THRESHOLD]          # answered by YOLO alone
lat = sorted(x["ms"] for x in res)
golden = {
    "n": len(res),
    "accuracy_all": rate(res, lambda x: x["pred"] == x["label"]),
    "recall_defective_all": rate(defective, lambda x: x["pred"] == "defective"),
    "false_reject_rate_all": rate(ok, lambda x: x["pred"] == "defective"),
    "fast_path_share": rate(res, lambda x: x["conf"] >= KEEP_THRESHOLD),
    "escalation_rate": rate(res, lambda x: x["conf"] < KEEP_THRESHOLD),
    "accuracy_on_fast_path": rate(kept, lambda x: x["pred"] == x["label"]),
    "defects_missed_on_fast_path": sum(1 for x in kept if x["label"] == "defective" and x["pred"] == "ok"),
    "p95_cpu_latency_ms": lat[int(0.95 * (len(lat) - 1))],
    "note": "Escalation here counts confidence only; the app also escalates new parts (label count) and unknown classes.",
}
print(json.dumps(golden, indent=2))

**Read this before the demo.** `defects_missed_on_fast_path` is the number that matters on a
line: a defective part that YOLO passed with high confidence never reaches ER 2. If it is not 0,
raise `KEEP_THRESHOLD` (in the app: `EscalationPolicy.keep_threshold`), re-run this cell, and
report the trade-off (more escalation, more cost) honestly.

## Package the outputs

In [ ]:
import shutil
OUT = pathlib.Path("/content/out"); OUT.mkdir(exist_ok=True)
shutil.copy(BEST, OUT / "best.pt")
n_train = sum(1 for _ in (DATA / "splits/classify/train").rglob("*.jp*g"))
(OUT / "yolo_label_counts.json").write_text(json.dumps({PART_NUMBER: n_train}))
(OUT / "yolo_baseline.json").write_text(json.dumps(
    {"name": "baseline", "top1": float(val.top1), "epochs": EPOCHS, "imgsz": IMGSZ,
     "weights": "runs/baseline/weights/best.pt"}, indent=2))
(OUT / "golden_yolo.json").write_text(json.dumps({"summary": golden, "rows": res}, indent=2))
shutil.make_archive("/content/senpaiqc_yolo", "zip", OUT)
from google.colab import files
files.download("/content/senpaiqc_yolo.zip")

## Use the weights in the app (on your machine)

```bash
# repo root
mkdir -p runs/baseline/weights eval/results
unzip senpaiqc_yolo.zip -d /tmp/kl_yolo
mv /tmp/kl_yolo/best.pt runs/baseline/weights/best.pt
mv /tmp/kl_yolo/yolo_label_counts.json runs/baseline/
mv /tmp/kl_yolo/yolo_baseline.json /tmp/kl_yolo/golden_yolo.json eval/results/
pip install ultralytics            # already in server/api/requirements.txt
```

Then in `.env`:

```
YOLO_WEIGHTS=runs/baseline/weights/best.pt
YOLO_LABEL_COUNTS=runs/baseline/yolo_label_counts.json
```

Without `YOLO_LABEL_COUNTS` the policy sees 0 labels for the part and escalates **everything**
to ER 2 as a "new part". Then run `python eval/run_all.py` for the full metrics.

## Optional: teacher-student retrain

After inspectors confirm ER 2 decisions in the app, export them
(`POST /api/learn/labels/export?part_number=IMP-200`), zip the export's `classify/` folder as
`labels_classify.zip`, and run the two cells below. Then compare escalation with
`python eval/run_all.py --weights-after runs/retrain/weights/best.pt`.

In [ ]:
from google.colab import files
up = files.upload()   # labels_classify.zip
EXTRA = pathlib.Path("/content/extra")
with zipfile.ZipFile("/content/" + next(iter(up))) as z:
    z.extractall(EXTRA)
merged = DATA / "splits/classify_retrain"
shutil.rmtree(merged, ignore_errors=True)
shutil.copytree(DATA / "splits/classify", merged)
added = 0
for img in EXTRA.rglob("*.jpg"):
    cls = "ok" if img.parent.name == "ok" else "defective"   # same mapping as eval/train_yolo.py
    shutil.copy2(img, merged / "train" / cls / f"confirmed_{img.name}")
    added += 1
print("added", added, "confirmed labels")

In [ ]:
YOLO("yolo11n-cls.pt").train(data=str(merged), epochs=EPOCHS, imgsz=IMGSZ, batch=64, seed=0,
                             project="/content/runs", name="retrain", exist_ok=True)
files.download("/content/runs/retrain/weights/best.pt")